In [1]:
# ==========================================
# 1. Import Libraries
# ==========================================

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# ==========================================
# 2. Load Dataset
# ==========================================

data = fetch_california_housing()
X = data.data
y = data.target


# ==========================================
# 3. Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ==========================================
# 4. Define Models
# ==========================================

models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ),

    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR())
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor())
    ])
}


# ==========================================
# 5. Cross-Validation Setup
# ==========================================

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "r2": "r2",
    "neg_mse": "neg_mean_squared_error",
    "neg_mae": "neg_mean_absolute_error"
}


# ==========================================
# 6. Cross-Validation Comparison
# ==========================================

print("Cross Validation Results (5-Fold):\n")

cv_results = {}

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train,
                            cv=cv,
                            scoring=scoring,
                            return_train_score=False)

    r2_mean = scores["test_r2"].mean()
    r2_std = scores["test_r2"].std()

    mse_mean = -scores["test_neg_mse"].mean()
    mae_mean = -scores["test_neg_mae"].mean()

    cv_results[name] = r2_mean

    print(f"{name}")
    print(f"  R2:  {r2_mean:.4f} ± {r2_std:.4f}")
    print(f"  MSE: {mse_mean:.4f}")
    print(f"  MAE: {mae_mean:.4f}")
    print("-" * 40)


# ==========================================
# 7. Select Best Model (Based on R2)
# ==========================================

best_model_name = max(cv_results, key=cv_results.get)
best_model = models[best_model_name]

print("\nBest Model (based on CV R2):", best_model_name)


# ==========================================
# 8. Train Best Model on Full Training Data
# ==========================================

best_model.fit(X_train, y_train)


# ==========================================
# 9. Evaluate on Test Set
# ==========================================

y_pred = best_model.predict(X_test)

print("\nTest Set Performance:")
print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))


# ==========================================
# 10. Results Table
# ==========================================

results_df = pd.DataFrame({
    "Model": cv_results.keys(),
    "CV Mean R2": cv_results.values()
}).sort_values(by="CV Mean R2", ascending=False)

print("\nModel Comparison Table:\n")
print(results_df)


Cross Validation Results (5-Fold):

Linear Regression
  R2:  0.6115 ± 0.0124
  MSE: 0.5193
  MAE: 0.5291
----------------------------------------
Decision Tree
  R2:  0.5973 ± 0.0240
  MSE: 0.5383
  MAE: 0.4738
----------------------------------------
Random Forest
  R2:  0.8056 ± 0.0065
  MSE: 0.2599
  MAE: 0.3338
----------------------------------------
SVR
  R2:  0.7365 ± 0.0103
  MSE: 0.3521
  MAE: 0.3951
----------------------------------------
KNN
  R2:  0.6836 ± 0.0091
  MSE: 0.4227
  MAE: 0.4442
----------------------------------------

Best Model (based on CV R2): Random Forest

Test Set Performance:
R2: 0.8061857564039718
MSE: 0.2539759249192041
MAE: 0.32681185043604677

Model Comparison Table:

               Model  CV Mean R2
2      Random Forest    0.805606
3                SVR    0.736537
4                KNN    0.683628
0  Linear Regression    0.611457
1      Decision Tree    0.597252
